# Phase 01.01 — Dataset contract & split freeze

This notebook reads the official `data/train` and `data/public_test` directories, normalizes labeled training annotations, validates both video sets, prevents train/validation group leakage, and freezes validation sample IDs. Public-test rows are validated separately and are never included in the train/validation split.

In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path("/workspace/RoadBuddy")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("CC", "/usr/bin/gcc")
os.environ.setdefault("CXX", "/usr/bin/g++")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from roadbuddy_common import *

os.chdir(PROJECT_ROOT)
seed_everything(SEED)
ensure_dirs()
print("Project root:", PROJECT_ROOT)
print("Model revision:", MODEL_REVISION)

Project root: /workspace/RoadBuddy
Model revision: b98f263eab246eb5269ade64edbdca8a887dc44d


## 1. Official data configuration

The dataset root contains `train/` and `public_test/`. Only labeled `train/train.json` is split into train/validation. Public test is checked against its sample submission but never used for training or validation.

In [2]:
DATA_ROOT = PROJECT_ROOT / "data"
TRAIN_FILE = DATA_ROOT / "train" / "train.json"
PUBLIC_TEST_FILE = DATA_ROOT / "public_test" / "public_test.json"
PUBLIC_TEST_SUBMISSION_FILE = DATA_ROOT / "public_test" / "public_test_sample_submission.csv"
VALIDATION_FILE = None
VIDEO_ROOT = DATA_ROOT
VALIDATION_FRACTION = 0.20
CHECK_VIDEO_DECODE = False  # Set True for a slower full decode smoke check.
ALLOW_SPLIT_REPLACEMENT = False

SCHEMA_OVERRIDES = {
    # "sample_id": "your_id_column",
    # "group_id": "video_id",
    # "video_path": "video",
    # "question": "question",
    # "answer": "label",
    # "choices": "choices",
}

In [3]:
assert (DATA_ROOT / "train" / "videos").is_dir(), "Missing data/train/videos"
assert (DATA_ROOT / "public_test" / "videos").is_dir(), "Missing data/public_test/videos"
assert TRAIN_FILE.is_file(), f"Training annotation file not found: {TRAIN_FILE}"
assert PUBLIC_TEST_FILE.is_file(), f"Public-test annotation file not found: {PUBLIC_TEST_FILE}"
assert PUBLIC_TEST_SUBMISSION_FILE.is_file(), f"Sample submission not found: {PUBLIC_TEST_SUBMISSION_FILE}"

raw_train = load_table(TRAIN_FILE)
raw_public_test = load_table(PUBLIC_TEST_FILE)
public_submission = pd.read_csv(PUBLIC_TEST_SUBMISSION_FILE)

required_train = {"id", "question", "choices", "answer", "video_path"}
required_public = {"id", "question", "choices", "video_path"}
assert required_train <= set(raw_train.columns), f"Missing train columns: {sorted(required_train - set(raw_train.columns))}"
assert required_public <= set(raw_public_test.columns), f"Missing public-test columns: {sorted(required_public - set(raw_public_test.columns))}"
assert list(public_submission.columns) == ["id", "answer"]
assert raw_train.id.is_unique and raw_public_test.id.is_unique and public_submission.id.is_unique
assert set(raw_train.id).isdisjoint(set(raw_public_test.id))
assert raw_public_test.id.astype(str).tolist() == public_submission.id.astype(str).tolist(), "Public-test IDs/order differ from sample submission"

print("Raw train rows:", len(raw_train))
print("Raw public-test rows:", len(raw_public_test))
display(raw_train.head(3))
display(raw_public_test.head(3))

Raw train rows: 1490
Raw public-test rows: 405


,id,question,choices,answer,support_frames,video_path,_unused_
0,train_0001,Nếu xe ô tô đang chạy ở làn ngoài cùng bên phả...,"[A. Đúng, B. Sai]",B. Sai,[4.427402],train/videos/2b840c67_386_clip_002_0008_0018_Y...,cbb77f7bf70be7d60ac580753f67ee61@1
1,train_0002,Phần đường trong video cho phép các phương tiệ...,"[A. Đi thẳng, B. Đi thẳng và rẽ phải, C. Đi th...","C. Đi thẳng, rẽ trái và rẽ phải",[5.344766],train/videos/2b840c67_386_clip_002_0008_0018_Y...,cbb77f7bf70be7d60ac580753f67ee61@2
2,train_0003,Biển chỉ dẫn 3 hướng di chuyển chính tiếp theo...,"[A. Đúng, B. Sai]",A. Đúng,[3.845463],train/videos/fe716b14_386_clip_003_0018_0024_N...,cbb77f7bf70be7d60ac580753f67ee61@3


,id,question,choices,video_path
0,testa_0001,"Theo trong video, nếu ô tô đi hướng chếch sang...","[A. Không có thông tin, B. Dầu Giây Long Thành...",public_test/videos/efc9909e_908_clip_001_0000_...
1,testa_0002,"Theo trong video, nếu ô tô đi hướng chếch sang...","[A. Đúng, B. Sai]",public_test/videos/efc9909e_908_clip_001_0000_...
2,testa_0003,Trong video có xuất hiện biển cảnh báo không?,"[A. Có, B. Không]",public_test/videos/7194eae2_908_clip_004_0020_...


## 2. Normalize to the canonical contract


In [4]:
normalized_train = normalize_dataset(raw_train, video_root=VIDEO_ROOT, schema_overrides=SCHEMA_OVERRIDES)

if VALIDATION_FILE is not None:
    assert Path(VALIDATION_FILE).is_file(), f"Validation file not found: {VALIDATION_FILE}"
    normalized_val = normalize_dataset(load_table(Path(VALIDATION_FILE)), video_root=VIDEO_ROOT, schema_overrides=SCHEMA_OVERRIDES)
    train_df, val_df = normalized_train, normalized_val
    overlap = set(train_df.group_id) & set(val_df.group_id)
    assert not overlap, f"Provided train/validation files leak group IDs: {sorted(overlap)[:10]}"
    source_files = [str(TRAIN_FILE), str(VALIDATION_FILE)]
else:
    train_df, val_df = freeze_group_split(normalized_train, VALIDATION_FRACTION, SEED)
    source_files = [str(TRAIN_FILE)]

assert set(train_df.sample_id).isdisjoint(set(val_df.sample_id))
assert set(train_df.group_id).isdisjoint(set(val_df.group_id))
display(pd.DataFrame({
    "split": ["train", "validation"],
    "rows": [len(train_df), len(val_df)],
    "groups": [train_df.group_id.nunique(), val_df.group_id.nunique()],
}))


,split,rows,groups
0,train,1192,439
1,validation,298,110


## 3. Validate paths and labels


In [5]:
all_rows = pd.concat([train_df, val_df], ignore_index=True)
assert set(all_rows.answer.unique()) <= set(CHOICES)
path_report = validate_video_paths(all_rows, check_decode=CHECK_VIDEO_DECODE)
missing_videos = path_report.loc[~path_report.exists, "video_path"].tolist()
assert not missing_videos, f"Missing {len(missing_videos)} train videos; examples: {missing_videos[:10]}"
if CHECK_VIDEO_DECODE:
    bad_decode = path_report.loc[path_report.decodes != True, "video_path"].tolist()
    assert not bad_decode, f"Undecodable train videos: {bad_decode[:10]}"

public_choice_counts = raw_public_test.choices.map(lambda values: len(dict.fromkeys(map(str, values))) if isinstance(values, list) else 0)
assert public_choice_counts.between(2, 4).all(), "Public-test choices must contain 2-4 distinct values"
public_video_rows = pd.DataFrame({
    "video_path": raw_public_test.video_path.map(lambda value: str((VIDEO_ROOT / str(value)).resolve()))
})
public_path_report = validate_video_paths(public_video_rows, check_decode=CHECK_VIDEO_DECODE)
missing_public_videos = public_path_report.loc[~public_path_report.exists, "video_path"].tolist()
assert not missing_public_videos, f"Missing {len(missing_public_videos)} public-test videos; examples: {missing_public_videos[:10]}"
if CHECK_VIDEO_DECODE:
    bad_public_decode = public_path_report.loc[public_path_report.decodes != True, "video_path"].tolist()
    assert not bad_public_decode, f"Undecodable public-test videos: {bad_public_decode[:10]}"

display(path_report.head())
display(public_path_report.head())
print("Training dataset contract: PASS")
print("Public-test dataset contract: PASS")

,video_path,exists,decodes
0,/workspace/RoadBuddy/data/train/videos/001a9a8...,True,None
1,/workspace/RoadBuddy/data/train/videos/00af5b3...,True,None
2,/workspace/RoadBuddy/data/train/videos/00b9d4a...,True,None
3,/workspace/RoadBuddy/data/train/videos/00e85ec...,True,None
4,/workspace/RoadBuddy/data/train/videos/0190262...,True,None


,video_path,exists,decodes
0,/workspace/RoadBuddy/data/public_test/videos/0...,True,None
1,/workspace/RoadBuddy/data/public_test/videos/0...,True,None
2,/workspace/RoadBuddy/data/public_test/videos/0...,True,None
3,/workspace/RoadBuddy/data/public_test/videos/0...,True,None
4,/workspace/RoadBuddy/data/public_test/videos/0...,True,None


Training dataset contract: PASS
Public-test dataset contract: PASS


## 4. Freeze artifacts

The immutable membership file is the cross-phase source of truth. Existing membership must match exactly unless replacement is explicitly authorized in the configuration cell.


In [6]:
split_dir = PATHS.phase1_split
ids_path = split_dir / "validation_sample_ids.json"
new_ids = sorted(val_df.sample_id.astype(str).tolist())

if ids_path.exists() and not ALLOW_SPLIT_REPLACEMENT:
    old_ids = json.loads(ids_path.read_text(encoding="utf-8"))
    assert old_ids == new_ids, "Frozen validation membership changed. Inspect the cause; do not silently overwrite it."

train_df.to_csv(split_dir / "train.csv", index=False)
val_df.to_csv(split_dir / "validation.csv", index=False)
path_report.to_csv(split_dir / "video_path_report.csv", index=False)
public_path_report.to_csv(split_dir / "public_test_video_path_report.csv", index=False)
save_json(ids_path, new_ids)
save_json(split_dir / "manifest.json", split_manifest(train_df, val_df, source_files))

print("Saved:", split_dir)
print("Validation IDs:", len(new_ids))

Saved: /workspace/RoadBuddy/data/splits/phase01
Validation IDs: 298


## Nhận xét sau lần chạy Phase 01.01

**Trạng thái:** PASS. Dataset contract, split freeze và kiểm tra public test đều hoàn tất.

### Quy mô và cấu trúc dữ liệu

- Train gốc: 1.490 câu hỏi, 549 video.
- Public test: 405 câu hỏi, 182 video; ID và thứ tự khớp sample submission.
- Split theo video/group với seed 42: 1.192 train rows thuộc 439 groups và 298 validation rows thuộc 110 groups.
- Sample-ID overlap giữa train/validation: 0; group/video overlap: 0.
- Không thiếu video trong cả train lẫn public test.
- Phân bố nhãn train: A=433, B=396, C=239, D=124. Validation: A=86, B=100, C=70, D=42.
- Train có 273 câu 2 lựa chọn, 2 câu 3 lựa chọn và 917 câu 4 lựa chọn. Validation có 66 câu 2 lựa chọn và 232 câu 4 lựa chọn.

### Nhận định

Split hiện tại chống leakage ở cấp video tốt hơn random row split. Public test được kiểm tra riêng và không đi vào train/validation, nên không có test leakage. Loader đã xử lý JSON wrapper, `choices`, `support_frames`, lựa chọn lặp và tiền tố lựa chọn sai trong dữ liệu nguồn.

### Hạn chế cần ghi nhớ

- `CHECK_VIDEO_DECODE=False`: notebook mới xác nhận file tồn tại, chưa decode toàn bộ 731 video. Trước full run nên bật decode hoặc chạy một batch decode audit riêng.
- Tất cả 1.490 rows hiện có `question_type=unknown`. Vì vậy mọi phân tích theo question type ở Phase01.05 chưa có giá trị phân nhóm.
- Public test không có `answer`, nên không thể dùng để báo cáo accuracy/F1.
- `validation_sample_ids.json` là membership đã freeze. Chỉ thay split khi có quyết định thí nghiệm rõ ràng và đặt `ALLOW_SPLIT_REPLACEMENT=True` có chủ đích.